# 02_feature_eng
01_cleaning의 clean_data.csv를 입력으로 받아 모델링 직전 단계까지 처리

수행 작업:
1. 파생 변수 생성 (디지털 채택 지수, 영업 효율 등)
2. 순서형(Ordinal) 변수 → 숫자 그대로 유지 (RF는 인코딩 불필요)
3. 명목형(Nominal) 변수 → One-Hot Encoding
4. 창업 준비 변수 플래그 처리 (신규창업자 여부)
5. 최종 feature matrix (X)와 label (y) 저장 → data/processed/features.csv

In [ ]:
import sys, io
if sys.stdout.encoding != 'utf-8':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

In [ ]:
# 경로 설정 (노트북 기준: src/ 폴더에서 실행)
BASE_DIR = Path().resolve().parent
IN_PATH  = BASE_DIR / "data" / "processed" / "clean_data.csv"
OUT_PATH = BASE_DIR / "data" / "processed" / "features.csv"

print(f"IN_PATH  : {IN_PATH}")
print(f"OUT_PATH : {OUT_PATH}")

## 변수 유형 분류 (01_cleaning의 영문 별칭 기준)

In [ ]:
# 명목형: 숫자 코드이지만 크기에 의미 없음 → One-Hot
NOMINAL_COLS = [
    "age_group",      # 연령대: 1~5
    "location_type",  # 운영장소: 1~4
    "industry_l",     # 산업 대분류: 문자 코드 (C, G, I 등)
    "industry_m",     # 산업 중분류: 숫자 코드
    "prev_status",    # 직전종사상지위: 1~4
    "startup_motive", # 창업동기: 1~N
    "tenure_type",    # 점유형태: 1~3
    "policy_type1",   # 추진정책1: 0~N (0=미수혜)
    "disaster_aid1",  # 재난지원1: 0~N
]

# 순서형: 리커트 척도 등 크기에 의미 있음 → 숫자 그대로 사용
ORDINAL_COLS = [
    "imp_bizplan", "imp_market", "imp_exp", "imp_edu",  # 중요도 1~5
    "diff_location", "diff_sector", "diff_funding",     # 어려움 1~5
    "diff_tech", "diff_hr", "diff_admin", "diff_mgmt",
    "eval_vision", "eval_innov", "eval_risk", "eval_newtech",  # 운영평가 1~5
]

# 이진형: 0/1 (01_cleaning에서 이미 변환 완료)
BINARY_COLS = [
    "gender", "biz_type", "franchise", "startup_type", "reloc_exp",
    "ecommerce_yn", "debt_yn", "policy_exp", "policy_aware",
    "did_bizplan", "did_market", "did_exp", "did_edu",
    "digi1_kiosk", "digi2_robot", "digi3_delivery", "digi4_sns",
    "digi5_erp", "digi6_qrorder", "digi7_other", "digi8_none",
    "digi_intent",
]

# 연속형: 그대로 사용 (스케일링은 03_balancing.py의 SMOTE 직전에)
CONTINUOUS_COLS = [
    "total_workers", "startup_count", "store_area",
    "op_hours_daily", "op_days_monthly", "op_months_yearly",
    "prep_years", "prep_months", "startup_cost_total",
    "sales", "op_profit", "op_cost", "cost_salary", "cost_rent",
    "pay_cash_pct", "pay_card_pct", "pay_easy_pct",
]

## 데이터 로딩

In [ ]:
if not IN_PATH.exists():
    raise FileNotFoundError(
        f"입력 파일 없음: {IN_PATH}\n"
        "먼저 01_cleaning.ipynb를 실행하세요."
    )

df = pd.read_csv(IN_PATH, encoding="utf-8-sig", low_memory=False)
print(f"[로딩] {df.shape[0]:,}행 × {df.shape[1]}열")
df.head(3)

## 1. 파생 변수 생성

In [ ]:
def create_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    # 1-1. 디지털 도입 총 개수 (7개 항목 합산)
    digi_items = ["digi1_kiosk", "digi2_robot", "digi3_delivery",
                  "digi4_sns", "digi5_erp", "digi6_qrorder", "digi7_other"]
    existing = [c for c in digi_items if c in df.columns]
    df["digi_count"] = df[existing].sum(axis=1).astype(int)

    # 1-2. 디지털 완전 미대응 플래그
    if "digi8_none" in df.columns:
        df["digi_zero"] = ((df["digi8_none"] == 1) & (df["digi_count"] == 0)).astype(int)
    else:
        df["digi_zero"] = (df["digi_count"] == 0).astype(int)

    # 1-3. 영업이익률 = 영업이익 / 매출금액
    if "sales" in df.columns and "op_profit" in df.columns:
        df["profit_margin"] = np.where(
            df["sales"] > 0,
            df["op_profit"] / df["sales"],
            0.0
        ).clip(-1, 1)

    # 1-4. 1인당 매출
    if "sales" in df.columns and "total_workers" in df.columns:
        df["sales_per_worker"] = np.where(
            df["total_workers"] > 0,
            df["sales"] / df["total_workers"],
            df["sales"]
        )

    # 1-5. 창업 준비 활동 총 수
    prep_cols = ["did_bizplan", "did_market", "did_exp", "did_edu"]
    existing_prep = [c for c in prep_cols if c in df.columns]
    df["prep_activity_count"] = df[existing_prep].sum(axis=1)

    # 1-6. 연간 실제 영업일수
    if "op_days_monthly" in df.columns and "op_months_yearly" in df.columns:
        df["op_days_annual"] = df["op_days_monthly"] * df["op_months_yearly"]

    # 1-7. 신규창업자 플래그
    if "startup_type" in df.columns:
        df["is_new_startup"] = (df["startup_type"] == 1).astype(int)

    print("[파생변수 생성]")
    derived = ["digi_count", "digi_zero", "profit_margin",
               "sales_per_worker", "prep_activity_count", "op_days_annual", "is_new_startup"]
    for col in derived:
        if col in df.columns:
            print(f"  + {col}")

    return df


df = create_derived_features(df)

## 2. 순서형 변수 dtype 정리

In [ ]:
def fix_ordinal_dtype(df: pd.DataFrame) -> pd.DataFrame:
    """순서형 변수를 명시적으로 int 처리 (float → int)."""
    for col in ORDINAL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    return df


df = fix_ordinal_dtype(df)

## 3. 명목형 One-Hot Encoding

In [ ]:
def encode_nominal(df: pd.DataFrame) -> pd.DataFrame:
    """
    명목형 변수를 One-Hot으로 인코딩.
    결측은 'unknown' 범주로 처리 후 인코딩.
    """
    cols_to_encode = [c for c in NOMINAL_COLS if c in df.columns]

    for col in cols_to_encode:
        if df[col].dtype in [float, np.float64]:
            df[col] = df[col].fillna(0).astype(int).astype(str)
        else:
            df[col] = df[col].fillna("unknown").astype(str)

    df = pd.get_dummies(df, columns=cols_to_encode, prefix=cols_to_encode,
                        drop_first=False, dtype=int)

    encoded_new = [c for c in df.columns
                   if any(c.startswith(base + "_") for base in cols_to_encode)]
    print(f"[OHE] {len(cols_to_encode)}개 명목형 → {len(encoded_new)}개 더미 변수 생성")

    return df


df = encode_nominal(df)

## 4. Feature 목록 확정 및 요약 출력

In [ ]:
def build_feature_list(df: pd.DataFrame) -> list:
    exclude = {"survey_year", "weight", "region_cd", "target_raw", "target"}
    return [c for c in df.columns if c not in exclude]


def print_summary(df: pd.DataFrame, feature_cols: list) -> None:
    print("\n[최종 Feature Matrix 요약]")
    print(f"  전체 샘플: {len(df):,}건")
    print(f"  Feature 수: {len(feature_cols)}개")
    print(f"  target=1 (지속): {df['target'].sum():,}건 ({df['target'].mean()*100:.1f}%)")
    print(f"  target=0 (중단): {(df['target']==0).sum():,}건 "
          f"({(df['target']==0).mean()*100:.1f}%)")

    null_cnt = df[feature_cols].isnull().sum().sum()
    print(f"  전체 결측: {null_cnt}건 {'(없음)' if null_cnt == 0 else '← 확인 필요'}")

    print("\n[변수 카테고리별 feature 수]")
    cat_map = {}
    for c in feature_cols:
        if any(c.startswith(b + "_") for b in NOMINAL_COLS):
            cat_map[c] = "명목형(OHE)"
        elif c in ORDINAL_COLS:
            cat_map[c] = "순서형"
        elif c in BINARY_COLS:
            cat_map[c] = "이진형"
        elif c in CONTINUOUS_COLS:
            cat_map[c] = "연속형"
        else:
            cat_map[c] = "파생변수"
    counter = Counter(cat_map.values())
    for k, v in sorted(counter.items()):
        print(f"  {k}: {v}개")


feature_cols = build_feature_list(df)
print_summary(df, feature_cols)

## 5. 저장

In [ ]:
save_cols = feature_cols + ["target"]
for meta in ["survey_year", "weight"]:
    if meta in df.columns:
        save_cols = [meta] + save_cols

df[save_cols].to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[저장 완료] {OUT_PATH}")
print(f"  shape: {len(df):,}행 × {len(save_cols)}열")

# feature 목록을 별도 텍스트로 저장 (03_balancing에서 참조)
feat_list_path = OUT_PATH.parent / "feature_list.txt"
with open(feat_list_path, "w", encoding="utf-8") as f:
    f.write("\n".join(feature_cols))
print(f"  feature 목록: {feat_list_path} ({len(feature_cols)}개)")